# Exercise 7: Capstone — End-to-End Research Agent

**Level:** Capstone

This is the final exercise. You will build a complete, end-to-end research agent that combines everything from the previous exercises: chains, memory, graphs, conditional routing, multi-agent patterns, and tool use.

The agent accepts a research topic, gathers information using tools, analyzes it with an LLM, generates a structured report, and includes a human-in-the-loop approval step.

**Minimal scaffolding is provided. You are the builder now.**

## 1. Setup

In [ ]:
!pip install langgraph langchain langchain-google-genai langchain-community -q

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "your-gemini-key-here"

## 2. Architecture Overview

```
START
  │
  ▼
[Plan Research]  ← Decides what data to gather
  │
  ▼
[Gather Data]    ← Uses tools to collect information
  │
  ▼
[Analyze]        ← LLM synthesizes findings
  │
  ▼
[Generate Report]← Structured output (Pydantic)
  │
  ▼
[Human Review]   ← interrupt_before for approval
  │
  ├── approved → [Finalize] → END
  └── revise   → [Analyze]  (loop back)
```

## 3. Define the Tools

These simulate external data sources. In production, these would call real APIs.

In [ ]:
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for information about a topic. Returns relevant snippets."""
    # Simulated search results
    results = {
        "ai aviation": (
            "1. Airlines are investing $4.2B in AI by 2025 (IATA report).\n"
            "2. AI-powered predictive maintenance reduces downtime by 35%.\n"
            "3. Chatbot adoption in airlines grew 45% in 2024.\n"
            "4. Travel tech companies reported 20% efficiency gains from AI-driven pricing.\n"
            "5. Delta Airlines saved $200M annually through AI fuel optimization."
        ),
        "sustainable aviation": (
            "1. SAF production reached 300M liters globally in 2024.\n"
            "2. EU mandates 6% SAF blend by 2030.\n"
            "3. SAF costs 2-4x more than conventional jet fuel.\n"
            "4. Airlines committed $30B to SAF investment through 2030.\n"
            "5. Hydrogen-powered aircraft expected by 2035 (Airbus ZEROe)."
        ),
    }
    # Fuzzy match
    query_lower = query.lower()
    for key, value in results.items():
        if key in query_lower or any(word in query_lower for word in key.split()):
            return value
    return f"Search results for '{query}': General information found. The topic is actively discussed in industry publications."

@tool
def get_market_data(sector: str) -> str:
    """Get market data and statistics for a given sector."""
    data = {
        "aviation": "Market size: $876B (2024). Growth: 5.2% CAGR. Key players: Sabre, Travelport, Travelsky. Digital transformation spending: $35B.",
        "ai": "Market size: $196B (2024). Growth: 37% CAGR. Enterprise adoption: 72%. Key use cases: NLP, computer vision, predictive analytics.",
        "travel": "Market size: $1.1T (2024). Online booking: 65%. Mobile booking: 48%. Personalization impact: +23% conversion.",
    }
    sector_lower = sector.lower()
    for key, value in data.items():
        if key in sector_lower:
            return value
    return f"Market data for {sector}: Sector is growing. Detailed data requires premium access."

@tool
def get_expert_opinions(topic: str) -> str:
    """Retrieve expert opinions and industry perspectives on a topic."""
    return (
        f"Expert opinions on '{topic}':\n"
        f"- Dr. Sarah Chen (MIT): 'AI agents will handle 60% of customer interactions by 2027.'\n"
        f"- Marco Rossi (Aviation Tech CTO): 'The integration of LLMs into travel tech is the biggest shift since online booking.'\n"
        f"- Julia Park (McKinsey): 'Companies that adopt AI agents early will see 3x ROI within 18 months.'"
    )

tools = [search_web, get_market_data, get_expert_opinions]

# Test the tools
print(search_web.invoke("ai aviation"))
print("\n" + get_market_data.invoke("aviation"))
print("\n" + get_expert_opinions.invoke("AI in aviation"))

## 4. Define State and Report Schema

In [ ]:
from typing import TypedDict, Annotated, Optional
from pydantic import BaseModel, Field
import operator

class ResearchReport(BaseModel):
    """Structured research report."""
    title: str = Field(description="Report title")
    executive_summary: str = Field(description="2-3 sentence executive summary")
    key_findings: list[str] = Field(description="3-5 key findings with data points")
    market_context: str = Field(description="Market size, growth, and competitive landscape")
    expert_perspectives: list[str] = Field(description="2-3 expert quotes or perspectives")
    challenges: list[str] = Field(description="2-3 challenges or risks")
    recommendations: list[str] = Field(description="2-3 actionable recommendations")
    confidence_level: str = Field(description="HIGH, MEDIUM, or LOW — based on data quality")

class AgentState(TypedDict):
    topic: str
    research_plan: str
    search_results: str
    market_data: str
    expert_data: str
    analysis: str
    report: Optional[dict]        # Serialized ResearchReport
    report_text: str              # Formatted report text
    human_decision: str           # "approved" or "revise"
    revision_feedback: str
    iteration: int
    log: Annotated[list[str], operator.add]

print("State and schema defined.")

## 5. Build the Agent Nodes

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.3)
creative_model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)
parser = StrOutputParser()

def plan_research(state: AgentState) -> dict:
    """Plan what data to gather."""
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a research planner. Given a topic, output a brief research plan "
         "listing what information to gather. Be specific about data points needed."),
        ("human", "Plan research for: {topic}")
    ])
    chain = prompt | model | parser
    plan = chain.invoke({"topic": state["topic"]})
    print(f"  [Planner] Research plan created")
    return {"research_plan": plan, "log": ["Research planned"]}

def gather_data(state: AgentState) -> dict:
    """Use tools to collect information."""
    topic = state["topic"]
    
    # Call all three tools
    search = search_web.invoke(topic)
    market = get_market_data.invoke(topic)
    experts = get_expert_opinions.invoke(topic)
    
    print(f"  [Gatherer] Collected data from 3 sources")
    return {
        "search_results": search,
        "market_data": market,
        "expert_data": experts,
        "log": ["Data gathered from 3 tools"]
    }

def analyze_data(state: AgentState) -> dict:
    """Synthesize all gathered data."""
    iteration = state.get("iteration", 0) + 1
    
    revision_context = ""
    if iteration > 1 and state.get("revision_feedback"):
        revision_context = f"\n\nPREVIOUS FEEDBACK TO ADDRESS:\n{state['revision_feedback']}"
    
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a research analyst. Synthesize the provided data into a coherent analysis. "
         "Identify patterns, draw conclusions, and note any data gaps or uncertainties."
         "{revision_context}"),
        ("human",
         "Topic: {topic}\n\n"
         "Web research:\n{search}\n\n"
         "Market data:\n{market}\n\n"
         "Expert opinions:\n{experts}\n\n"
         "Produce a thorough analysis.")
    ])
    chain = prompt | model | parser
    analysis = chain.invoke({
        "topic": state["topic"],
        "search": state["search_results"],
        "market": state["market_data"],
        "experts": state["expert_data"],
        "revision_context": revision_context
    })
    
    print(f"  [Analyst] Analysis complete (iteration {iteration})")
    return {"analysis": analysis, "iteration": iteration, "log": [f"Analysis v{iteration} complete"]}

def generate_report(state: AgentState) -> dict:
    """Generate a structured report."""
    report_model = model.with_structured_output(ResearchReport)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a senior research analyst. Generate a comprehensive, structured report. "
         "Use specific data points from the analysis. Be precise and actionable."),
        ("human",
         "Topic: {topic}\n\nAnalysis:\n{analysis}\n\n"
         "Raw data sources:\n{search}\n{market}\n{experts}\n\n"
         "Generate the final research report.")
    ])
    chain = prompt | report_model
    report = chain.invoke({
        "topic": state["topic"],
        "analysis": state["analysis"],
        "search": state["search_results"],
        "market": state["market_data"],
        "experts": state["expert_data"]
    })
    
    # Format as readable text
    report_text = format_report(report)
    
    print(f"  [Reporter] Report generated: {report.title}")
    return {
        "report": report.model_dump(),
        "report_text": report_text,
        "log": ["Structured report generated"]
    }

def format_report(report: ResearchReport) -> str:
    """Format a ResearchReport into readable text."""
    sections = [
        f"# {report.title}",
        f"\n## Executive Summary\n{report.executive_summary}",
        f"\n## Key Findings",
        *[f"- {f}" for f in report.key_findings],
        f"\n## Market Context\n{report.market_context}",
        f"\n## Expert Perspectives",
        *[f"- {p}" for p in report.expert_perspectives],
        f"\n## Challenges",
        *[f"- {c}" for c in report.challenges],
        f"\n## Recommendations",
        *[f"- {r}" for r in report.recommendations],
        f"\n---\nConfidence Level: {report.confidence_level}"
    ]
    return "\n".join(sections)

print("All agent nodes defined.")

## 6. Human-in-the-Loop Review

This node presents the report and waits for human input.

In [ ]:
def human_review(state: AgentState) -> dict:
    """Present report for human review."""
    print("\n" + "=" * 60)
    print("REPORT FOR REVIEW")
    print("=" * 60)
    print(state["report_text"])
    print("=" * 60)
    
    # In Colab, use input(). In production, this would be an API callback.
    decision = input("\nApprove this report? (yes/no): ").strip().lower()
    
    if decision in ["yes", "y", "approve"]:
        return {
            "human_decision": "approved",
            "log": ["Human approved the report"]
        }
    else:
        feedback = input("Feedback for revision: ").strip()
        return {
            "human_decision": "revise",
            "revision_feedback": feedback,
            "log": [f"Human requested revision: {feedback[:50]}..."]
        }

def finalize_report(state: AgentState) -> dict:
    """Finalize the approved report."""
    print(f"  [Finalize] Report approved and finalized after {state['iteration']} iteration(s)")
    return {"log": ["Report finalized and delivered"]}

print("Review nodes defined.")

## 7. Build the Complete Graph

In [ ]:
from langgraph.graph import StateGraph, START, END

def review_router(state: AgentState) -> str:
    if state["human_decision"] == "approved":
        return "approved"
    if state["iteration"] >= 3:
        print("  [Router] Max iterations — auto-approving")
        return "approved"
    return "revise"

# Build the graph
builder = StateGraph(AgentState)

builder.add_node("plan", plan_research)
builder.add_node("gather", gather_data)
builder.add_node("analyze", analyze_data)
builder.add_node("report", generate_report)
builder.add_node("review", human_review)
builder.add_node("finalize", finalize_report)

# Edges
builder.add_edge(START, "plan")
builder.add_edge("plan", "gather")
builder.add_edge("gather", "analyze")
builder.add_edge("analyze", "report")
builder.add_edge("report", "review")

# Conditional: review → finalize or review → analyze (loop)
builder.add_conditional_edges(
    "review",
    review_router,
    {
        "approved": "finalize",
        "revise": "analyze"
    }
)

builder.add_edge("finalize", END)

capstone_graph = builder.compile()
print("Capstone graph compiled!")

In [ ]:
# Visualize
print(capstone_graph.get_graph().draw_mermaid())

## 8. Run the Full Agent

When the human review step appears, you can approve or request revisions.

In [ ]:
print("Starting research agent...\n")

result = capstone_graph.invoke({
    "topic": "AI-powered agents in the aviation industry",
    "research_plan": "",
    "search_results": "",
    "market_data": "",
    "expert_data": "",
    "analysis": "",
    "report": None,
    "report_text": "",
    "human_decision": "",
    "revision_feedback": "",
    "iteration": 0,
    "log": []
})

In [ ]:
# View the execution log
print("Execution Log:")
for i, entry in enumerate(result["log"], 1):
    print(f"  {i}. {entry}")

print(f"\nTotal iterations: {result['iteration']}")
print(f"Decision: {result['human_decision']}")

In [ ]:
# Print the final report
print(result["report_text"])

## 9. Auto-Approve Version (for testing)

For automated testing, here is a version without the interactive input.

In [ ]:
def auto_review(state: AgentState) -> dict:
    """Automatically approve the report (for testing)."""
    print(f"  [Auto-Review] Auto-approving report")
    return {"human_decision": "approved", "log": ["Auto-approved for testing"]}

# Build auto-approve version
auto_builder = StateGraph(AgentState)
auto_builder.add_node("plan", plan_research)
auto_builder.add_node("gather", gather_data)
auto_builder.add_node("analyze", analyze_data)
auto_builder.add_node("report", generate_report)
auto_builder.add_node("review", auto_review)
auto_builder.add_node("finalize", finalize_report)

auto_builder.add_edge(START, "plan")
auto_builder.add_edge("plan", "gather")
auto_builder.add_edge("gather", "analyze")
auto_builder.add_edge("analyze", "report")
auto_builder.add_edge("report", "review")
auto_builder.add_conditional_edges("review", review_router, {
    "approved": "finalize",
    "revise": "analyze"
})
auto_builder.add_edge("finalize", END)

auto_graph = auto_builder.compile()

# Run
print("Running auto-approve version...\n")
result = auto_graph.invoke({
    "topic": "Sustainable aviation fuel and the future of green air travel",
    "research_plan": "",
    "search_results": "",
    "market_data": "",
    "expert_data": "",
    "analysis": "",
    "report": None,
    "report_text": "",
    "human_decision": "",
    "revision_feedback": "",
    "iteration": 0,
    "log": []
})

print("\n" + result["report_text"])

---
## YOUR TURN: Extend the Capstone

Choose one or more of these challenges:

### Challenge A: Add Real Tool Calling
Replace the simulated tools with LangChain's tool-calling agent pattern. The LLM should decide *which* tools to call and *what arguments* to pass, rather than calling all tools every time.

### Challenge B: Add a Competitive Analysis Node
Add a node that compares the topic against competitors. The node should produce a comparison table (as structured output) that gets included in the report.

### Challenge C: Add Checkpointing
Use LangGraph's `MemorySaver` checkpointer so that the graph can be interrupted and resumed. This is critical for the human-in-the-loop pattern in production.

### Challenge D: Build Your Own Agent
Design and build a completely different agent for a domain you care about. Use the patterns from all 7 exercises.

In [ ]:
# YOUR TURN: Choose a challenge and build it.
#
# No scaffolding this time. You have all the patterns.
# Design it. Build it. Test it.

## Concepts Covered Across All Exercises

| Exercise | Concepts |
|----------|----------|
| 01 - MCP Server | Tools, Resources, MCP protocol, `@mcp.tool()` |
| 02 - First Chain | ChatGoogleGenerativeAI, PromptTemplate, LCEL, StrOutputParser, Pydantic |
| 03 - Memory | ChatMessageHistory, RunnableWithMessageHistory, message trimming |
| 04 - Basic Graph | StateGraph, TypedDict, nodes, edges, compile, invoke, visualize |
| 05 - Conditional Graph | add_conditional_edges, route functions, loops, should_continue |
| 06 - Multi-Agent | Specialized agents, feedback loops, shared state orchestration |
| 07 - Capstone | End-to-end agent, tools, structured output, human-in-the-loop |

You now have the building blocks to design and build production AI agents.